In [1]:
from pathlib import Path
from project_paths import paths
import polars as pl
from imd_features.config import FeatureSetConfig
from imd_features.evaluate import evaluate

output_dir = paths.output

In [2]:
config_path = output_dir / "mixed_reduction_b78f17cd_config.json"
config = FeatureSetConfig.model_validate_json(config_path.read_text())
df = pl.read_parquet(output_dir / f"{config.output_name}.parquet")

In [3]:
results = evaluate(df, config)

In [4]:
performance = pl.DataFrame([
    {
        "model": model_name,
        "r2": f"{r['r2_mean']:.3f} ± {r['r2_std']:.3f}",
        "rmse": f"{r['rmse_mean']:.1f} ± {r['rmse_std']:.1f}",
        "spearman": f"{r['spearman_mean']:.3f} ± {r['spearman_std']:.3f}",
    }
    for model_name, r in results.items()
])
performance

model,r2,rmse,spearman
str,str,str,str
"""ridge""","""0.510 ± 0.393""","""10.2 ± 3.6""","""0.824 ± 0.072"""
"""random_forest""","""0.813 ± 0.015""","""6.8 ± 0.5""","""0.910 ± 0.014"""


In [5]:
rf_importance = pl.DataFrame(results["random_forest"]["feature_importance"]).sort(
    "importance_mean", descending=True
)
rf_importance

feature,group,importance_mean,importance_std
str,str,f64,f64
"""total_sfw_claims""","""universal_credit""",0.110773,0.01511
"""total_prepfw_claims""","""universal_credit""",0.105805,0.023434
"""mean_monthly_claims""","""universal_credit""",0.099145,0.011738
"""total_planfw_claims""","""universal_credit""",0.098342,0.006178
"""lsoa_max_price""","""land_registry""",0.091232,0.032858
…,…,…,…
"""landuse_storage_0""","""osm_landuse""",6.2672e-9,1.1050e-8
"""%_claims_nwr""","""universal_credit""",0.0,0.0
"""%_claims_planfw""","""universal_credit""",0.0,0.0


In [6]:
ridge_importance = pl.DataFrame(results["ridge"]["feature_importance"]).sort(
    "importance_mean", descending=True
)
ridge_importance

feature,group,importance_mean,importance_std
str,str,f64,f64
"""bicycle-theft""","""crime""",3.855542,0.979725
"""violent-crime""","""crime""",3.572072,0.918893
"""vehicle-crime""","""crime""",3.192886,0.626774
"""lsoa_average_price""","""land_registry""",3.149316,1.626067
"""pension_age_population""","""population""",2.675442,0.397973
…,…,…,…
"""landuse_depot_0""","""osm_landuse""",0.071255,0.065087
"""%_claims_nwr""","""universal_credit""",0.0,0.0
"""%_claims_planfw""","""universal_credit""",0.0,0.0


In [7]:
rf_groups = pl.DataFrame(results["random_forest"]["group_importance"]).sort(
    "total_importance", descending=True
)
ridge_groups = pl.DataFrame(results["ridge"]["group_importance"]).sort(
    "total_importance", descending=True
)
print("Random Forest:")
print(rf_groups)
print("\nRidge:")
print(ridge_groups)

Random Forest:
shape: (7, 4)
┌──────────────────┬──────────────────┬─────────────────┬────────────┐
│ group            ┆ total_importance ┆ mean_importance ┆ n_features │
│ ---              ┆ ---              ┆ ---             ┆ ---        │
│ str              ┆ f64              ┆ f64             ┆ i64        │
╞══════════════════╪══════════════════╪═════════════════╪════════════╡
│ universal_credit ┆ 0.57385          ┆ 0.057385        ┆ 10         │
│ land_registry    ┆ 0.220503         ┆ 0.016962        ┆ 13         │
│ osm_landuse      ┆ 0.064305         ┆ 0.002217        ┆ 29         │
│ osm_amenities    ┆ 0.057179         ┆ 0.003812        ┆ 15         │
│ crime            ┆ 0.041532         ┆ 0.002769        ┆ 15         │
│ population       ┆ 0.028733         ┆ 0.007183        ┆ 4          │
│ connectivity     ┆ 0.013896         ┆ 0.003474        ┆ 4          │
└──────────────────┴──────────────────┴─────────────────┴────────────┘

Ridge:
shape: (7, 4)
┌──────────────────┬──────